In [1]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = "cuda" if torch.cuda.is_available() else "cpu"

Torch: 2.10.0+cu130
CUDA available: True
GPU: NVIDIA RTX 4500 Ada Generation


In [3]:
import transformers.modeling_utils as modeling_utils

# Workaround for transformers 4.51 + Python 3.13
if not hasattr(modeling_utils, "init_empty_weights"):
    from contextlib import nullcontext
    modeling_utils.init_empty_weights = nullcontext

MODEL_NAME = "ai-sage/Giga-Embeddings-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    trust_remote_code=True,
    device_map="auto"
)

model = model.to(device)
model.eval()

print("Model loaded successfully.")




'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 448d3ad2-6b2b-4577-a54f-7d20f5ab9f9f)')' thrown while requesting HEAD https://huggingface.co/ai-sage/Giga-Embeddings-instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully.


In [4]:
for name, module in model.named_modules():
    if "latent" in name.lower():
        print(name, type(module))

latent_attention_model <class 'transformers_modules.ai-sage.Giga-Embeddings-instruct.2cf0fdc97194aaedf10ac0e6bf798834acd31042.modeling_gigarembed.LatentAttentionModel'>
latent_attention_model.cross_attend_blocks <class 'torch.nn.modules.container.ModuleList'>
latent_attention_model.cross_attend_blocks.0 <class 'transformers_modules.ai-sage.Giga-Embeddings-instruct.2cf0fdc97194aaedf10ac0e6bf798834acd31042.modeling_gigarembed.Attention'>
latent_attention_model.cross_attend_blocks.0.to_q <class 'torch.nn.modules.linear.Linear'>
latent_attention_model.cross_attend_blocks.0.to_kv <class 'torch.nn.modules.linear.Linear'>
latent_attention_model.cross_attend_blocks.0.to_out <class 'torch.nn.modules.linear.Linear'>
latent_attention_model.cross_attend_blocks.1 <class 'transformers_modules.ai-sage.Giga-Embeddings-instruct.2cf0fdc97194aaedf10ac0e6bf798834acd31042.modeling_gigarembed.FeedForward'>
latent_attention_model.cross_attend_blocks.1.gate_proj <class 'torch.nn.modules.linear.Linear'>
latent

In [5]:
latent_outputs = {}

def save_hook(name):
    def hook(module, input, output):
        latent_outputs[name] = output.detach().float().cpu()
    return hook

# Register hooks
model.latent_attention_model.cross_attend_blocks[0].register_forward_hook(
    save_hook("latent_after_attention")
)

model.latent_attention_model.cross_attend_blocks[1].register_forward_hook(
    save_hook("latent_after_ffn")
)

print("Hooks registered.")


Hooks registered.
